# Comparación NLP de noticias sobre exportaciones colombianas

Esta muestra adapta el modelo público **PySentimiento / RoBERTuito** a noticias reales sobre exportaciones colombianas de bienes. Conserva como referencia separada la medición por diccionario descrita en BE1340. La ejecución no entrena un modelo nuevo ni reproduce resultados de precisión de otros autores.

La explicación se presenta primero en español para revisión. El código conserva identificadores en inglés.

## 1. Cargar el proyecto

En Colab, carga un ZIP de este repositorio privado descargado mientras tienes acceso. No escribas contraseñas ni tokens en las celdas. En Jupyter local, abre el notebook desde la raíz del proyecto. Los pesos y los cuerpos completos de las noticias no están dentro del ZIP.

In [ ]:
from pathlib import Path
import io
import os
import sys
import tempfile
import zipfile

try:
    from google.colab import files
except ImportError:
    repo = Path.cwd()
else:
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload exactly one project ZIP")
    destination = Path(tempfile.mkdtemp(prefix="trade_news_"))
    with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
        if sum(item.file_size for item in archive.infolist()) > 50_000_000:
            raise ValueError("Unexpectedly large project archive")
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive path")
            if (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Symbolic links are not accepted")
        archive.extractall(destination)
    roots = [path.parent for path in destination.rglob("README.md") if (path.parent / "trade_news").is_dir()]
    if len(roots) != 1:
        raise ValueError("Could not identify the project root")
    repo = roots[0]

assert (repo / "trade_news").is_dir(), "Run from the repository directory"
os.chdir(repo)
sys.path.insert(0, str(repo))
print("Project loaded:", repo.name)

## 2. Comprobar procedencia y resultados guardados

Esta sección funciona sin descargar pesos ni ejecutar inferencia. Reproduce la relación entre probabilidades, clases y agregados mensuales. Para repetir la clasificación desde el texto se necesita el entorno opcional y el archivo local de noticias. Las probabilidades son predicciones, no etiquetas humanas.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from trade_news.articles import read_jsonl
from trade_news.contextual import verify_scores, aggregate_scores

data = repo/"data"
rows = read_jsonl(data/"headlines.jsonl")
scores = read_jsonl(data/"contextual_scores.jsonl")
manifest = json.loads((data/"contextual_manifest.json").read_text("utf8"))
assert manifest["status"] == "complete"
verify_scores(data, scores, manifest)
months = [r["month"] for r in json.loads((data/"collection_log.json").read_text("utf8"))["requests"]]
contextual = {"all_eligible": aggregate_scores(scores, rows, months),
              "dictionary_matched": aggregate_scores(scores, rows, months, dictionary_matched=True)}
assert contextual == json.loads((data/"contextual_indices.json").read_text("utf8"))
validation = json.loads((data/"validation_report.json").read_text("utf8"))
print("Modelo:", manifest["model_id"], "| revisión:", manifest["model_revision"])
print("Artículos con predicciones:", len(scores), "| etiquetas humanas reales:", validation["n_labeled"])
print("Fragmentos del cuerpo procesados:", sum(r["body"]["chunks"] for r in scores))
display(pd.DataFrame(contextual["all_eligible"])[["month", "articles", "mean_probability_balance"]])

## 3. Comparar métodos sobre artículos con coincidencias en el diccionario

El puntaje de RoBERTuito es `P(positivo) − P(negativo)`. Para cada cuerpo se combinan probabilidades de fragmentos sin superposición mediante sus tokens; después se promedian los puntajes por artículo dentro del mes.

El método BE1340 agrupa ocurrencias de palabras antes de dividir `(P − N)/(P + N)`. No es la misma ponderación. Restringir el nuevo modelo a los artículos con coincidencias ayuda a examinar la influencia de la cobertura del vocabulario. Los artículos sin palabras positivas o negativas no aportan al numerador ni al denominador de sentimiento del diccionario. La comparación no establece cuál método es correcto.

La auditoría anterior verifica todo el archivo. La gráfica siguiente se restringe al marco principal directo de Valora/La República, igual que el tablero; Analdex permanece complementario.


In [ ]:
from trade_news.coverage import in_primary_press_frame
from trade_news.bank_method import aggregate
primary = [r for r in rows if r["eligible_for_index"] and in_primary_press_frame(r)]
bank = [r for r in aggregate(primary, months) if r["activity"] == "All exports"]
common = aggregate_scores(scores, primary, months, dictionary_matched=True)
figure = go.Figure()
figure.add_scatter(x=months, y=[r["sentiment"] for r in bank], mode="lines+markers", name="Diccionario", connectgaps=False)
figure.add_scatter(x=months, y=[r["mean_probability_balance"] for r in common], mode="lines+markers", name="RoBERTuito · subconjunto con coincidencias", connectgaps=False)
figure.update_layout(template="plotly_white", title="Marco principal · dos mediciones automáticas de tono", xaxis_title="Mes de publicación", yaxis_title="Puntaje")
figure.update_yaxes(range=[-1.05, 1.05], zeroline=False)
figure.show()


## 4. Inspeccionar el caso de reactivación de exportaciones

El titular y el cuerpo se procesan por separado. Una clase neutra puede tener un puntaje pequeño distinto de cero: la clase corresponde al máximo entre las tres probabilidades, mientras que el puntaje solo resta las probabilidades positiva y negativa. La probabilidad máxima no es precisión validada.

In [ ]:
lookup = {r["record_id"]: r for r in rows}
prediction_lookup = {r["record_id"]: r for r in scores}
record_id = "b30d353a15880d52e0b3"
example = lookup[record_id]
prediction = prediction_lookup[record_id]
print(example.get("publisher_title") or example["title"])
print(example["article_url"], example["publication_date"])
print("Diccionario sobre el cuerpo:", example["sentiment_score"], "| P:", example["positive_count"], "| N:", example["negative_count"])
table = []
for unit in ["headline", "body"]:
    value = prediction[unit]
    table.append({"texto": unit, "clase": value["label"], "puntaje": value["score"],
                  "P_negativo": value["probabilities"]["NEG"], "P_neutro": value["probabilities"]["NEU"],
                  "P_positivo": value["probabilities"]["POS"], "fragmentos": value["chunks"], "tokens": value["tokens"]})
display(pd.DataFrame(table))

## 5. Interpretación y evaluación
El modelo se entrenó con tuits en español. Su transferencia a noticias de exportaciones requiere referencias humanas: el paquete independiente fija 200 artículos y sigue sin adjudicación. La precisión de otros autores y las probabilidades del modelo no son precisión comprobada aquí. El caso conocido de reactivación es diagnóstico, está fuera de esa muestra y no se convierte en etiqueta humana.

La nueva recuperación tiene prensa en los 80 meses desde marzo de 2020; fija dos dominios y mantiene Analdex como complemento. Las propuestas de informe estadístico no prueban novedad y quedan sujetas a revisión. El archivo numérico de la investigadora permite una comparación retrospectiva real, pero no acreditó publicaciones/versiones históricas. La pequeña reducción de error al añadir tono coexiste con diagnósticos adversos; no valida un predictor.


In [ ]:
economic = json.loads((data/"economic_evaluation.json").read_text("utf8"))
main = economic["scenarios"][economic["primary_scenario"]]
print(economic["status"])
print(main["availability_status"])
display(pd.DataFrame(main["metrics"]).T)
print("Diagnósticos y conclusión: docs/ECONOMIC_RESULTS_ES.md")
independent = json.loads((data/"independent_evaluation.json").read_text("utf8"))
print("Muestra fijada:", independent["locked_sample_records"], "| tonos adjudicados:", independent["adjudicated_tone_labels"])


## 6. Repetir opcionalmente la inferencia

Desactivado por defecto. La descarga pública de los pesos requiere aproximadamente 415 MiB, además de las dependencias. Para procesar cuerpos completos se necesita el archivo local ignorado `data/raw/articles/`. Si no está presente, no se sustituye por titulares ni se generan predicciones ficticias. Repetir únicamente los agregados desde las probabilidades guardadas usa `python scripts/run_contextual.py --offline`.

In [ ]:
RUN_INFERENCE = False
if RUN_INFERENCE:
    import subprocess
    if not (repo/"data/raw/articles").is_dir():
        raise ValueError("La inferencia completa requiere el archivo local de artículos")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-contextual.txt"], check=True)
    subprocess.run([sys.executable, "scripts/download_contextual_model.py"], check=True)
    subprocess.run([sys.executable, "scripts/run_contextual.py", "--model-dir", "data/raw/contextual_model"], check=True)
else:
    print("Inferencia opcional desactivada: se usaron únicamente resultados guardados y verificados.")

## Referencias y créditos

- [PySentimiento: código público de sus autores](https://github.com/pysentimiento/pysentimiento).
- [Modelo RoBERTuito para sentimiento en español](https://huggingface.co/pysentimiento/robertuito-sentiment-analysis).
- [Dávila y Chasi: Índice de sentimiento económico, Banco Central del Ecuador](https://doi.org/10.47550/RCE/35.2.1). Aplicación económica relacionada; no se reutilizan sus etiquetas ni porcentajes de precisión.
- [Mora-Quiñones, Orozco-Gallo y Mora-Pérez: BE1340](https://doi.org/10.32468/be.1340). Referencia del método léxico.

Los pesos del modelo y el preprocesamiento reutilizado mantienen sus autores y restricciones de uso científico/no comercial. La selección del corpus, la segmentación de artículos largos y esta comparación son adaptaciones del proyecto. Consulta `docs/CONTEXTUAL_REPLICATION_ES.md` y `reference/PYSENTIMIENTO_LICENSE.txt`.